# Where does a shock go, and when?

A cross-sectional spatial model answers "where". It says a shock at one place
raises outcomes at its neighbours, and the multiplier is settled instantly
inside the single period the data covers. That is a strong claim about timing,
and it is usually made by accident rather than on purpose.

Space-time models separate the two questions. This notebook fits three of them
and ends with a decomposition rather than a regression:

1. **STAR** — all spatial transmission is *lagged*. Within a period units are
   independent given the past, so there is no simultaneity and no Jacobian.
2. **SpVAR** — several variables, each with its own equation, and a
   contemporaneous ordering the researcher must state. The ordering is not a
   nuisance parameter; it is the causal assumption.
3. **STPCA** — no model of transmission at all. It asks which spatial patterns
   account for the variance, and how they move over time.

As in notebook 06, every dataset is simulated from a stated equation, so each
estimate can be read against the number that generated it.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

using SpatialEconometrics
using LinearAlgebra, SparseArrays, Statistics, Printf, Random

## 1. STAR: diffusion that takes a period to arrive

The generating equation is

    y_t = phi0 * y_{t-1} + phi1 * W * y_{t-1} + X_t * b + c + e_t

so a shock at unit `i` in period `t` reaches `i` itself next period through
`phi0`, and reaches `i`'s neighbours next period through `phi1`. Nothing
crosses space within a period.

This is a different world from the SAR panel, where `rho * W * y_t` puts the
whole spatial multiplier inside the current period. Fitting a SAR panel to STAR
data does not fail loudly — it returns a `rho`, and that `rho` is an artefact of
forcing lagged diffusion through a contemporaneous channel. The second cell
below shows what it reports on data with no contemporaneous spatial term at all.

In [ ]:
RNG = MersenneTwister(2026)
side = 7
W = make_rook_weights(side)
N = size(W, 1)
T = 60

PHI0, PHI1, BETA, ICEPT = 0.40, 0.40, [1.0, -0.5], 0.25

function simulate_star(rng, W, N, T, phi0, phi1, beta, icept; burn = 50)
    A = phi0 .* Matrix(1.0I, N, N) .+ phi1 .* Matrix(W)
    k = length(beta)
    X = randn(rng, N * (T + burn), k)
    y = zeros(N * (T + burn))
    prev = zeros(N)
    for t in 1:(T + burn)
        r = ((t - 1) * N + 1):(t * N)
        cur = A * prev .+ X[r, :] * beta .+ icept .+ randn(rng, N)
        y[r] = cur
        prev = cur
    end
    keep = (burn * N + 1):((T + burn) * N)
    return y[keep], X[keep, :]
end

y, X = simulate_star(RNG, W, N, T, PHI0, PHI1, BETA, ICEPT)
st = fit_star(y, X, W, N, T; spatial_lags = 1, temporal_lags = 1)

@printf("units %d, periods %d\n\n", N, T)
@printf("%-24s %10s %10s %+9s\n", "", "estimate", "truth", "error")
@printf("%-24s %10.4f %10.4f %+9.4f\n", "phi[y(t-1)]",   st.phi[1, 1], PHI0, st.phi[1, 1] - PHI0)
@printf("%-24s %10.4f %10.4f %+9.4f\n", "phi[W y(t-1)]", st.phi[2, 1], PHI1, st.phi[2, 1] - PHI1)

### What a contemporaneous model reports on lagged data

There is no `rho` in the truth. Whatever comes back is the price of the wrong
timing assumption.

In [ ]:
sar = fit_spatial_panel(SAR(), NoFE(), W, X, y, N, T;
                        approach = DirectQML(), method = ExactLU())

@printf("true contemporaneous spatial effect : %.4f\n", 0.0)
@printf("rho from a SAR panel on STAR data   : %+.4f\n", sar.rho)
println()
println("The number is not small, and nothing in the output marks it as spurious.")

## 2. The impulse response, which is where the two lags combine

`space_time_irf` traces a unit shock forward. The response at horizon `h` is
governed by powers of `A = phi0*I + phi1*W`, so the own-effect and the spatial
effect interact: by horizon 2 a shock has reached neighbours-of-neighbours
through `phi1^2`, and has also come *back* to the origin through the same path.

The stability condition is on the spectral radius of `A`, not on either `phi`
separately. With `phi0 = phi1 = 0.4` and row-standardised `W`, the radius is
about 0.8 — stable, but slow to decay, which the cumulative column shows.

In [ ]:
A = PHI0 .* Matrix(1.0I, N, N) .+ PHI1 .* Matrix(W)
@printf("spectral radius of A : %.4f\n\n", maximum(abs, eigvals(A)))

H   = 6
irf = space_time_irf(st; horizon = H)
nbr = findall(!iszero, W[1, :])              # the neighbours of unit 1

own     = [irf.psi[h + 1][1, 1]        for h in 0:H]
nbr_avg = [mean(irf.psi[h + 1][1, nbr]) for h in 0:H]
cum_own = cumsum(own)

@printf("%8s %14s %18s %18s\n", "horizon", "own", "mean neighbour", "cumulative own")
for h in 0:H
    @printf("%8d %14.5f %18.5f %18.5f\n", h, own[h + 1], nbr_avg[h + 1], cum_own[h + 1])
end

# The sum of every Psi is the long-run multiplier, (I - A)^-1.
lr = inv(Matrix(1.0I, N, N) - A)
@printf("\nlong-run own effect from (I - A)^-1 : %.5f\n", mean(diag(lr)))
@printf("long-run total per unit             : %.5f\n", mean(sum(lr, dims = 2)))

## 3. SpVAR: the ordering *is* the assumption

Two variables per region — call them spending `g` and output `y` — each with
its own equation, each carrying a spatial lag. The question a VAR must answer
before it can be estimated is which variable may move the other *within* the
period. In a time-series VAR that choice is usually defended by appeal to
timing. Here it also decides whether a cross-border contemporaneous effect
exists, so it is doing twice the work.

`fit_spvar` therefore has **no default ordering**. The truth below is that
spending moves first: a shock to `g` reaches `y` in the same period, at home
through `p0` and across the border through `p1`, while a shock to `y` does not
reach `g` until the next period.

Fitting the reversed ordering is not a robustness check. It fits a different
model of the world, and the two log-likelihoods are not two estimates of one
quantity.

In [ ]:
function ring_W(n)
    I_ = Int[]; J_ = Int[]
    for i in 1:n
        push!(I_, i); push!(J_, mod1(i - 1, n))
        push!(I_, i); push!(J_, mod1(i + 1, n))
    end
    A = sparse(I_, J_, ones(length(I_)), n, n)
    return sparse(A ./ sum(A, dims = 2))
end

TRUTH = (r1 = 0.35, r2 = -0.20, p0 = 0.40, p1 = 0.15,
         a = ((0.30, 0.10), (0.15, -0.05), (-0.10, 0.08), (0.25, 0.12)))

function simulate_spvar(rng, Wl, N, T; burn = 200)
    Wd = Matrix(Wl[2]); Im = Matrix(1.0I, N, N); Z = zeros(N, N)
    t = TRUTH
    C0 = [Im - t.r1 * Wd              Z;
          -(t.p0 * Im + t.p1 * Wd)    Im - t.r2 * Wd]
    C1 = [t.a[1][1] * Im + t.a[1][2] * Wd   t.a[2][1] * Im + t.a[2][2] * Wd;
          t.a[3][1] * Im + t.a[3][2] * Wd   t.a[4][1] * Im + t.a[4][2] * Wd]
    F = lu(C0)
    rad = maximum(abs, eigvals(F \ C1))
    rad < 1 || error("the DGP is explosive, spectral radius $rad")
    yv = zeros(2N)
    Y = Matrix{Float64}(undef, N * T, 2)
    for tt in 1:(T + burn)
        yv = F \ (C1 * yv + randn(rng, 2N))
        if tt > burn
            k = tt - burn
            for i in 1:N
                Y[(k - 1) * N + i, 1] = yv[i]
                Y[(k - 1) * N + i, 2] = yv[N + i]
            end
        end
    end
    return Y, rad
end

Nv, Tv = 14, 400
Wv  = ring_W(Nv)
WLv = order_weights_list(Wv, 1)
Yv, rad = simulate_spvar(MersenneTwister(4242), WLv, Nv, Tv)

@printf("ring of %d regions, %d periods, 2 variables; DGP spectral radius %.4f\n\n",
        Nv, Tv, rad)

m_true = fit_spvar(Yv, Wv, Nv, Tv; ordering = ["g", "y"], s = 1, Wl = WLv,
                   varnames = ["g", "y"])
m_rev  = fit_spvar(Yv, Wv, Nv, Tv; ordering = ["y", "g"], s = 1, Wl = WLv,
                   varnames = ["g", "y"])

@printf("log-likelihood, g -> y : %12.3f\n", m_true.loglik)
@printf("log-likelihood, y -> g : %12.3f   (difference %.3f)\n",
        m_rev.loglik, m_true.loglik - m_rev.loglik)
println()
println("Recovery under the true ordering, truth in brackets:")
@printf("  eq g: rho_1  = %+.4f  [%+.2f]\n", m_true.rho[1][1],   TRUTH.r1)
@printf("  eq y: rho_1  = %+.4f  [%+.2f]\n", m_true.rho[2][1],   TRUTH.r2)
@printf("  eq y: g      = %+.4f  [%+.2f]   contemporaneous, at home\n",
        m_true.theta[2][1], TRUTH.p0)
@printf("  eq y: W.g    = %+.4f  [%+.2f]   contemporaneous, across the border\n",
        m_true.theta[2][2], TRUTH.p1)

## 4. Where a spending shock actually goes

`spvar_irf` returns the reduced-form responses: a vector of matrices, one per
horizon, each of size `(N*K) x (N*K)`, already multiplied through by the inverse
of the contemporaneous matrix. Rows and columns stack variable-major, so index
`i` is `g` at unit `i` and index `N + i` is `y` at unit `i`.

That layout is what makes the ordering visible. At horizon **0** the response of
`y` to a `g` shock is non-zero — that is `p0` and `p1` doing their work — while
the response of `g` to a `y` shock is exactly zero, because the model does not
allow it. Reverse the ordering and the zeros move.

In [ ]:
psi = spvar_irf(m_true; horizon = 6)

shock_g_at_1 = 1               # column: shock to g at unit 1
shock_y_at_1 = Nv + 1          # column: shock to y at unit 1
nbr_v = findall(!iszero, Wv[1, :])

@printf("%8s %14s %16s %18s %16s\n",
        "horizon", "g at unit 1", "y at unit 1", "y at neighbours", "g | y shock")
for h in 0:6
    P = psi[h + 1]
    @printf("%8d %14.5f %16.5f %18.5f %16.5f\n", h,
            P[1, shock_g_at_1],
            P[shock_y_at_1, shock_g_at_1],
            mean(P[Nv .+ nbr_v, shock_g_at_1]),
            P[1, shock_y_at_1])
end

println()
println("The last column is zero at horizon 0 by construction: under this")
println("ordering y cannot move g within the period. It is not an estimate.")

## 5. Is there any spatial interaction at all?

A SpVAR with `s = 0` is a panel VAR: each region's variables depend on their own
past, with no spatial terms anywhere. That is a nested model, so the comparison
is a likelihood-ratio test rather than a judgement call.

`lr_spatial_interaction` performs it, returning the statistic, its degrees of
freedom and a p-value. On data generated *with* spatial interaction it should
reject decisively — and a test that cannot reject on data built to be rejectable
is not worth running on real data.

In [ ]:
lr = lr_spatial_interaction(m_true, Yv, Wv)

@printf("LR statistic : %.3f\n", lr.stat)
@printf("df           : %d\n", lr.df)
@printf("p-value      : %.3e\n", lr.p_value)

# With s = 0 there are no spatial terms, so the weight list must shrink to
# match: fit_spvar requires exactly s + 1 matrices.
m_nospace = fit_spvar(Yv, Wv, Nv, Tv; ordering = ["g", "y"], s = 0,
                      Wl = WLv[1:1], varnames = ["g", "y"])

@printf("\nloglik with spatial terms   : %12.3f\n", m_true.loglik)
@printf("loglik without them (s = 0) : %12.3f\n", m_nospace.loglik)
@printf("twice the difference        : %12.3f   (the LR statistic above)\n",
        2 * (m_true.loglik - m_nospace.loglik))

## 6. STPCA: patterns without a transmission model

The three models so far all specify *how* a shock travels. Spatio-temporal
principal component analysis gives that up. It asks a descriptive question:
which spatial patterns account for the variance, and how do their amplitudes
move over time?

What makes it spatial rather than ordinary PCA is the objective. Each component
maximises the product of its variance and its spatial autocorrelation, so a
pattern that is high-variance but spatially incoherent scores poorly, and so
does a smooth pattern that barely varies. The eigenvalue is literally that
product, which the check below verifies with `moran_coefficient` computed
independently of the decomposition.

Two fields are planted on purpose — the smoothest pattern the lattice admits and
the most contrasting one — on different temporal profiles, into features 1 and
3. Feature 2 carries neither and exists to be ignored.

In [ ]:
ts = collect(0.5:1.0:16.5); nt = length(ts)
IV = (0.0, 17.0)
Wp = row_standardize(make_rook_weights(6)); Np = size(Wp, 1)
p  = 3

se     = spatial_eigenvectors(Wp)
smooth = se.vectors[:, 1]
rough  = se.vectors[:, end]

rngp = MersenneTwister(2026)
Xp = Array{Float64}(undef, Np, p, nt)
for i in 1:Np, j in 1:p
    Xp[i, j, :] = randn(rngp) .* cos.(ts ./ 4) .+ 0.5 .* randn(rngp, nt)
end
for i in 1:Np
    Xp[i, 1, :] .+= 5.0 * smooth[i] .* sin.(2pi .* ts ./ 17)
    Xp[i, 3, :] .+= 5.0 * rough[i]  .* sin.(4pi .* ts ./ 17)
end

@printf("planted smooth field, Moran I   = %+.4f\n", moran_coefficient(smooth, Wp))
@printf("planted contrast field, Moran I = %+.4f\n", moran_coefficient(rough, Wp))
println()

mp = stpca(Xp, Wp; measure = :moran, n_basis = 5, times = ts, interval = IV,
           varnames = ["smooth-carrier", "noise", "contrast-carrier"])

println("eigenvalue = Var(score) x Moran I(score), checked independently:")
@printf("%6s %14s %12s %12s %14s\n", "comp", "eigenvalue", "Var", "I", "Var x I")
for k in (1, 2, length(mp.values) - 1, length(mp.values))
    s = mp.scores[:, k]
    v = var(s, corrected = false)
    i_ = moran_coefficient(s, Wp)
    @printf("%6d %14.6f %12.6f %+12.6f %14.6f\n", k, mp.values[k], v, i_, v * i_)
end

## 7. What to carry away

- **Timing is an assumption, not a detail.** A SAR panel fitted to STAR data
  returned a contemporaneous `rho` of about +0.24 on a truth where that
  parameter is exactly zero. Nothing in the output flags it. The only defence is
  deciding, before estimating, whether diffusion within a period is plausible
  for the process at hand.
- The STAR stability condition is on the spectral radius of
  `phi0*I + phi1*W` — about 0.8 here — and not on either coefficient. Two
  individually modest lags can still produce a long-lived response, which the
  cumulative column makes visible.
- **A SpVAR ordering is the causal claim.** Reversing it changed the
  log-likelihood by 11 points, but the two fits are not rival estimates of one
  quantity: under one ordering the contemporaneous effect of `g` on `y` is a
  parameter, under the other it does not exist. Choose the ordering from
  institutional knowledge, report it, and do not search over it.
- The reduced-form IRF makes the ordering legible: at horizon 0 the response of
  `g` to a `y` shock is exactly 0.00000 because the model forbids it, while `y`
  responds to `g` immediately both at home and across the border.
- Whether there is *any* spatial interaction is a nested test, not a judgement:
  the LR statistic of 1228.5 on 7 degrees of freedom equals twice the
  log-likelihood difference exactly, which is a useful internal check that the
  restricted model really is the same model restricted.
- STPCA's eigenvalue is the product of variance and spatial autocorrelation, and
  the table above recomputes both halves with `moran_coefficient` to confirm it.
  It describes patterns; it does not model transmission, and it cannot be used
  to argue that one place influenced another.